In [8]:
import os
import re
import subprocess
import tempfile

# Define paths
input_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio"
output_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch"
temp_dir = tempfile.mkdtemp()

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Get list of all files in input directory
all_files = os.listdir(input_dir)

# Parse participant numbers and group files
participant_files = {}

for file in all_files:
    if file.endswith('.wav'):
        match = re.search(r'participant_([0-9]+)_design_(looming|tactile).wav', file)
        if match:
            participant_id = match.group(1)
            design_type = match.group(2)
            
            # Initialize dictionary entry if not exists
            if participant_id not in participant_files:
                participant_files[participant_id] = {}
                
            participant_files[participant_id][design_type] = file

print(f"Found files for {len(participant_files)} participants")

# Function to process audio files for each participant
def process_participant(participant_id, file_dict):
    # Skip if we don't have both looming and tactile files
    if 'looming' not in file_dict or 'tactile' not in file_dict:
        print(f"Skipping participant {participant_id} - missing files")
        return False
    
    looming_file = os.path.join(input_dir, file_dict['looming'])
    tactile_file = os.path.join(input_dir, file_dict['tactile'])
    output_file = os.path.join(output_dir, f"participant_{participant_id}_combined.wav")
    
    try:
        # Create a single FFmpeg command that does everything:
        # 1. Load both files
        # 2. Extract first channel from each
        # 3. Resample to 44100Hz
        # 4. Combine into stereo
        # 5. Output as WAV file
        
        cmd = [
            "ffmpeg",
            "-y",  # Overwrite output if exists
            "-i", looming_file,
            "-i", tactile_file,
            "-filter_complex", 
            "[0:a]channelsplit=channel_layout=stereo:channels=0,aresample=44100[left]; [1:a]channelsplit=channel_layout=stereo:channels=0,aresample=44100[right]; [left][right]amerge=inputs=2[aout]",
            "-map", "[aout]",
            "-c:a", "pcm_s16le",  # Standard PCM format
            "-ar", "44100",       # Sample rate 44100Hz
            "-ac", "2",           # Stereo output
            output_file
        ]
        
        print(f"Running FFmpeg...")
        result = subprocess.run(cmd, capture_output=True, text=True)
        
        if result.returncode != 0:
            print(f"FFmpeg error: {result.stderr}")
            
            # Try alternate approach with simplified filter
            print("Trying alternate approach...")
            cmd_alt = [
                "ffmpeg",
                "-y",
                "-i", looming_file,
                "-i", tactile_file,
                "-filter_complex", 
                "[0:a]pan=mono|c0=c0,aresample=44100[left]; [1:a]pan=mono|c0=c0,aresample=44100[right]; [left][right]join=inputs=2:channel_layout=stereo[aout]",
                "-map", "[aout]",
                "-c:a", "pcm_s16le",
                "-ar", "44100",
                output_file
            ]
            
            result_alt = subprocess.run(cmd_alt, capture_output=True, text=True)
            
            if result_alt.returncode != 0:
                # If that fails too, try an even simpler approach
                print("Trying simple approach...")
                cmd_simple = [
                    "ffmpeg",
                    "-y",
                    "-i", looming_file,
                    "-i", tactile_file,
                    "-filter_complex", 
                    "[0:a]aresample=44100[l];[1:a]aresample=44100[r];[l][r]amerge=inputs=2[aout]",
                    "-map", "[aout]",
                    "-ac", "2",
                    "-ar", "44100",
                    output_file
                ]
                result_simple = subprocess.run(cmd_simple, capture_output=True, text=True)
                
                if result_simple.returncode != 0:
                    print(f"All FFmpeg approaches failed: {result_simple.stderr}")
                    return False
        
        print(f"Successfully processed participant {participant_id}")
        return True
        
    except Exception as e:
        print(f"Error processing participant {participant_id}: {str(e)}")
        return False

# Process all participants
success_count = 0
total_count = len(participant_files)
print(f"Processing {total_count} participants...")

for participant_id, file_dict in participant_files.items():
    print(f"\nProcessing participant {participant_id} ({success_count+1}/{total_count})")
    if process_participant(participant_id, file_dict):
        success_count += 1

print(f"\nProcessing complete. Successfully processed {success_count} out of {total_count} participants.")

# Check first output file
output_files = [f for f in os.listdir(output_dir) if f.endswith('.wav')]
if output_files:
    first_file = os.path.join(output_dir, output_files[0])
    print(f"\nFirst output file created: {first_file}")

Found files for 100 participants
Processing 100 participants...

Processing participant 00 (1/100)
Running FFmpeg...
FFmpeg error: ffmpeg version 6.1 Copyright (c) 2000-2023 the FFmpeg developers
  built with clang version 17.0.4
  configuration: --prefix=/d/bld/ffmpeg_1699729642246/_h_env/Library --cc=clang.exe --cxx=clang++.exe --nm=llvm-nm --ar=llvm-ar --disable-doc --disable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --ld=lld-link --target-os=win64 --enable-cross-compile --toolchain=msvc --host-cc=clang.exe --extra-libs=ucrt.lib --extra-libs=vcruntime.lib --extra-libs=oldnames.lib --strip=llvm-strip --disable-stripping --host-extralibs= --enable-gpl --enable-libx264 --enable-libx265 --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib --enable-libopus --pkg-config=/d/bld/ffmpeg_1699729642246/_build_env/Library/b

In [2]:
"""
Combine “looming” + “tactile” WAVs into stereo with
a 5‑burst calibration beep in front.

Requires: FFmpeg 6.x in PATH  •  Python ≥3.8
"""

from __future__ import annotations
import os, re, shutil, subprocess, tempfile

# ─── 1. FOLDERS ────────────────────────────────────────────────────
INPUT_DIR  = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio"
OUTPUT_DIR = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CALIB_FILE = os.path.join(OUTPUT_DIR, "calibration_beep.wav")
TMP_DIR = tempfile.mkdtemp(prefix="bs_audio_")
print(f"Temp folder   : {TMP_DIR}\nCalibration   : {CALIB_FILE}")

# ─── 2. tiny helper ────────────────────────────────────────────────
def run(cmd: list[str]) -> None:
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode:
        raise RuntimeError(f"FFmpeg ERROR\n{' '.join(cmd)}\n{p.stderr.strip()}")

# ─── 3. build calibration clip (once) ──────────────────────────────
def build_calibration_clip(path: str) -> None:
    if os.path.exists(path):
        print("✓  calibration clip already exists – reusing it")
        return

    filt = (
        "[0:a]volume=0.25[tone];"              # scale to −12 dBFS
        "[tone][1:a]concat=n=2:v=0:a=1[seg];"  # 20 ms tone + 80 ms silence
        "[seg]aloop=loop=4:size=4410[mono];"   # 4 extra plays → 5 total
        "[mono]pan=stereo|c0=c0|c1=c0[out]"    # duplicate to stereo
    )

    run([
        "ffmpeg","-y",
        # 0) 20 ms 1 kHz sine
        "-f","lavfi","-i","sine=frequency=1000:duration=0.02:sample_rate=44100",
        # 1) 80 ms silence
        "-f","lavfi","-i","anullsrc=r=44100:cl=mono:d=0.08",
        "-filter_complex",filt,
        "-map","[out]","-ar","44100","-ac","2", path
    ])
    print("✓  calibration clip created")

build_calibration_clip(CALIB_FILE)

# ─── 4. collect files ──────────────────────────────────────────────
participants: dict[str, dict[str,str]] = {}
pat = re.compile(r"participant_(\d+)_design_(looming|tactile)\.wav", re.I)
for fn in os.listdir(INPUT_DIR):
    m = pat.fullmatch(fn)
    if m:
        pid, kind = m.groups()
        participants.setdefault(pid, {})[kind.lower()] = fn
print(f"Found {len(participants)} participants\n")

# ─── 5. process one participant ────────────────────────────────────
def process(pid: str, files: dict[str,str]) -> None:
    if {"looming","tactile"} - files.keys():
        print(f"· skip {pid} – missing file(s)")
        return

    looming = os.path.join(INPUT_DIR, files["looming"])
    tactile = os.path.join(INPUT_DIR, files["tactile"])
    merge   = os.path.join(TMP_DIR,  f"{pid}_merge.wav")
    final   = os.path.join(OUTPUT_DIR,f"participant_{pid}_combined.wav")

    # 1) mono → stereo (L=looming, R=tactile)
    filt = (
        "[0:a]aresample=44100,pan=mono|c0=c0[L];"
        "[1:a]aresample=44100,pan=mono|c0=c0[R];"
        "[L][R]amerge=2[out]"
    )
    run([
        "ffmpeg","-y","-i",looming,"-i",tactile,
        "-filter_complex",filt,"-map","[out]",
        "-ar","44100","-ac","2","-c:a","pcm_s16le", merge
    ])

    # 2) prepend beep
    run([
        "ffmpeg","-y","-i",CALIB_FILE,"-i",merge,
        "-filter_complex","[0:a][1:a]concat=n=2:v=0:a=1[out]",
        "-map","[out]","-c:a","pcm_s16le", final
    ])
    os.remove(merge)
    print(f"✓  participant {pid} → {os.path.basename(final)}")

# ─── 6. batch loop ─────────────────────────────────────────────────
fails = 0
for n,(pid,fdict) in enumerate(sorted(participants.items()),1):
    try:
        print(f"[{n:3}/{len(participants)}] {pid}")
        process(pid, fdict)
    except Exception as e:
        fails += 1
        print(f"✗  {pid} failed:\n   {e}")

print(f"\nDone – {len(participants)-fails}/{len(participants)} OK")

# ─── 7. tidy temp dir (calib file stays) ───────────────────────────
shutil.rmtree(TMP_DIR, ignore_errors=True)
print("Scratch files removed.")


Temp folder   : C:\Users\COGPSY~1\AppData\Local\Temp\bs_audio_w7mjvevq
Calibration   : C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch\calibration_beep.wav
✓  calibration clip created
Found 100 participants

[  1/100] 00
✓  participant 00 → participant_00_combined.wav
[  2/100] 1
✓  participant 1 → participant_1_combined.wav
[  3/100] 10
✓  participant 10 → participant_10_combined.wav
[  4/100] 11
✓  participant 11 → participant_11_combined.wav
[  5/100] 12
✓  participant 12 → participant_12_combined.wav
[  6/100] 13
✓  participant 13 → participant_13_combined.wav
[  7/100] 14
✓  participant 14 → participant_14_combined.wav
[  8/100] 15
✓  participant 15 → participant_15_combined.wav
[  9/100] 16
✓  participant 16 → participant_16_combined.wav
[ 10/100] 17
✓  participant 17 → participant_17_combined.wav
[ 11/100] 18
✓  participant 18 → participant_18_combined.wav
[ 12/100] 19
✓  participant 19 → participant_19_combined.wav
[ 13/100] 2
✓ 

In [3]:
"""
Breathing‑Space Audio Assembler
────────────────────────────────
• Builds a 5‑burst calibration clip
  (20 ms beep + 80 ms silence, 5×, +20 dB gain)
  **every time** the script runs.
• Combines mono “looming” + “tactile” WAVs into a stereo WAV
  (L = looming, R = tactile) and prepends the calibration beeps.

Requires:  FFmpeg in PATH, Python ≥3.8
"""

from __future__ import annotations
import os
import re
import shutil
import subprocess
import tempfile


# ─────────────────────────── PATHS ─────────────────────────────
INPUT_DIR  = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio"
OUTPUT_DIR = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch"
os.makedirs(OUTPUT_DIR, exist_ok=True)

CALIB_FILE = os.path.join(OUTPUT_DIR, "calibration_beep.wav")
TMP_DIR    = tempfile.mkdtemp(prefix="bs_audio_")
print(f"Temp folder        : {TMP_DIR}\nCalibration target : {CALIB_FILE}\n")


# ──────────────────── helper: run ffmpeg & raise ───────────────
def run(cmd: list[str]) -> None:
    """Execute *cmd*; raise RuntimeError if FFmpeg returns non‑zero."""
    proc = subprocess.run(cmd, capture_output=True, text=True)
    if proc.returncode:
        raise RuntimeError(
            "FFmpeg failed\n" + " ".join(cmd) + "\n" + proc.stderr.strip()
        )


# ─────────── build the 5‑burst calibration WAV (overwrite) ────
def build_calibration(path: str) -> None:
    """
    Writes a 0.5 s stereo WAV:
        [20 ms 1 kHz sine @ +20 dB] + [80 ms silence] × 5
    """
    filter_graph = (
        "[0:a]volume=10[beep];"                    # +20 dB gain
        "[beep][1:a]concat=n=2:v=0:a=1[seg];"      # one 100 ms piece
        "[seg]aloop=loop=4:size=4410[mono];"       # repeat 4 extra times
        "[mono]pan=stereo|c0=c0|c1=c0[out]"        # mono → stereo
    )

    run([
        "ffmpeg", "-y",                             # -y = overwrite
        "-f", "lavfi", "-i", "sine=f=1000:d=0.02:r=44100",
        "-f", "lavfi", "-i", "anullsrc=r=44100:cl=mono:d=0.08",
        "-filter_complex", filter_graph,
        "-map", "[out]", "-ar", "44100", "-ac", "2", path,
    ])
    print("✓  calibration clip (re)created")


build_calibration(CALIB_FILE)


# ─────────── scan input dir and group by participant ──────────
pattern = re.compile(r"participant_(\d+)_design_(looming|tactile)\.wav", re.I)
participants: dict[str, dict[str, str]] = {}

for fn in os.listdir(INPUT_DIR):
    if not fn.lower().endswith(".wav"):
        continue
    m = pattern.fullmatch(fn)
    if m:
        pid, kind = m.groups()
        participants.setdefault(pid, {})[kind.lower()] = fn

print(f"Found {len(participants)} participants\n")


# ───────────── process a single participant’s files ───────────
def process(pid: str, files: dict[str, str]) -> None:
    if {"looming", "tactile"} - files.keys():
        print(f"· skip {pid} – missing looming/tactile WAV")
        return

    looming_in = os.path.join(INPUT_DIR, files["looming"])
    tactile_in = os.path.join(INPUT_DIR, files["tactile"])
    merge_tmp  = os.path.join(TMP_DIR, f"{pid}_merge.wav")
    final_out  = os.path.join(OUTPUT_DIR, f"participant_{pid}_combined.wav")

    # 1) merge mono → stereo (L = looming, R = tactile)
    merge_graph = (
        "[0:a]aresample=44100,pan=mono|c0=c0[L];"
        "[1:a]aresample=44100,pan=mono|c0=c0[R];"
        "[L][R]amerge=inputs=2[out]"
    )
    run([
        "ffmpeg", "-y",
        "-i", looming_in, "-i", tactile_in,
        "-filter_complex", merge_graph,
        "-map", "[out]", "-ar", "44100", "-ac", "2",
        "-c:a", "pcm_s16le", merge_tmp,
    ])

    # 2) prepend calibration clip
    run([
        "ffmpeg", "-y",
        "-i", CALIB_FILE, "-i", merge_tmp,
        "-filter_complex", "[0:a][1:a]concat=n=2:v=0:a=1[out]",
        "-map", "[out]", "-c:a", "pcm_s16le", final_out,
    ])

    os.remove(merge_tmp)
    print(f"✓  participant {pid} → {os.path.basename(final_out)}")


# ───────────────────── batch loop ──────────────────────────────
failed = 0
for n, (pid, fdict) in enumerate(sorted(participants.items()), 1):
    try:
        print(f"[{n:3}/{len(participants)}] {pid}")
        process(pid, fdict)
    except Exception as err:
        failed += 1
        print(f"✗  {pid} failed → {err}")

print(f"\nFinished – {len(participants) - failed}/{len(participants)} succeeded.")

# remove scratch dir (calibration file stays)
shutil.rmtree(TMP_DIR, ignore_errors=True)
print("Temporary merge files removed.")


Temp folder        : C:\Users\COGPSY~1\AppData\Local\Temp\bs_audio_rgnqamne
Calibration target : C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level1_AudioGeneration\ExperimentAudio_2ch\calibration_beep.wav

✓  calibration clip (re)created
Found 100 participants

[  1/100] 00
✓  participant 00 → participant_00_combined.wav
[  2/100] 1
✓  participant 1 → participant_1_combined.wav
[  3/100] 10
✓  participant 10 → participant_10_combined.wav
[  4/100] 11
✓  participant 11 → participant_11_combined.wav
[  5/100] 12
✓  participant 12 → participant_12_combined.wav
[  6/100] 13
✓  participant 13 → participant_13_combined.wav
[  7/100] 14
✓  participant 14 → participant_14_combined.wav
[  8/100] 15
✓  participant 15 → participant_15_combined.wav
[  9/100] 16
✓  participant 16 → participant_16_combined.wav
[ 10/100] 17
✓  participant 17 → participant_17_combined.wav
[ 11/100] 18
✓  participant 18 → participant_18_combined.wav
[ 12/100] 19
✓  participant 19 → participant_19_combined.wav